# Predicting Early Readmission of Diabetic Patients Using Machine Learning

**Authors:** Abdifatah Hussein, Isihack Harerimana, Shamel Iqbal  
**Course:** DATA 4382 — Spring 2026  
**Instructor:** Professor Rostami

## Complementary Experiments

One of the key findings from our baseline modeling phase was that the Random Forest model performed worse on older patients compared to younger ones — suggesting that age-related clinical patterns may differ significantly across patient groups. This notebook investigates two complementary strategies for improving prediction by tailoring models to subpopulations:

1. **Age-stratified classification** — train separate models on distinct patient age groups
2. **Cluster-based classification** — use KMeans clustering to discover natural subgroups in the data and train a dedicated model on each

Both strategies are evaluated against our best-performing baseline (Random Forest with undersampling, ROC-AUC = 0.65) to determine whether subgroup-specific modeling can close the performance gap.

### Setup and Library Imports

In [ ]:
# Enable automatic module reloading for iterative development
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from src.preprocessing.helpers import describe_dataset
from src.evaluation import *

In [ ]:
# Standardize plot styling for consistent, readable visuals
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams.update({'font.size': 15})
%config InlineBackend.figure_format='retina'

### Load Preprocessed Data

We use the train/test data saved from the preprocessing pipeline. Because undersampling produced the strongest results in our initial modeling phase, we apply the same undersampling strategy here before running any experiments.

In [ ]:
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test  = pd.read_csv('../data/y_test.csv')

print("Loaded preprocessed datasets.")

In [ ]:
# Apply undersampling to address the class imbalance (88% non-readmitted vs 12% readmitted)
X_train, y_train = undersample(X_train, y_train)

describe_dataset(X_train, X_test, y_train, y_test)

---

## Experiment 1 — Age-Stratified Classification

Our baseline results indicated a performance gap between younger and older patient groups. To address this, we partition the data into four age bands and train a dedicated Random Forest model on each band. The intuition is that different age groups may have distinct clinical patterns that a single general model cannot capture simultaneously.

**Age group definitions** (based on the encoded `age` integer from preprocessing):

| Group | Age Range | Encoded Values |
|-------|-----------|---------------|
| 1 | 0 – 49 years | 0, 1, 2, 3, 4 |
| 2 | 50 – 69 years | 5, 6 |
| 3 | 70 – 79 years | 7 |
| 4 | 80 – 100 years | 8, 9 |

In [ ]:
# Visualize age distribution to confirm appropriate group boundaries
counts, bins = np.histogram(X_train['age'])
plt.figure()
plt.hist(bins[:-1], bins, weights=counts, color='steelblue', edgecolor='white')
plt.title('Distribution of Patient Age Groups (Training Set)')
plt.xlabel('Age (encoded)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Helper functions to filter rows by age threshold
def age_below(X, y, threshold):
    mask = X['age'] < threshold
    return X[mask], y.loc[X[mask].index]

def age_above(X, y, threshold):
    mask = X['age'] > threshold
    return X[mask], y.loc[X[mask].index]

# Partition training data into four age bands
X_train_g1, y_train_g1 = age_below(X_train, y_train, 5)          # Group 1: <50
X_train_g2, y_train_g2 = age_above(*age_below(X_train, y_train, 7), 4)  # Group 2: 50–69
X_train_g3, y_train_g3 = age_above(*age_below(X_train, y_train, 8), 6)  # Group 3: 70–79
X_train_g4, y_train_g4 = age_above(X_train, y_train, 7)          # Group 4: 80+

# Mirror the same split on test data
X_test_g1, y_test_g1 = age_below(X_test, y_test, 5)
X_test_g2, y_test_g2 = age_above(*age_below(X_test, y_test, 7), 4)
X_test_g3, y_test_g3 = age_above(*age_below(X_test, y_test, 8), 6)
X_test_g4, y_test_g4 = age_above(X_test, y_test, 7)

print("Training records per age group:")
print(f"  Group 1 (0–49):   {X_train_g1.shape[0]:,}")
print(f"  Group 2 (50–69):  {X_train_g2.shape[0]:,}")
print(f"  Group 3 (70–79):  {X_train_g3.shape[0]:,}")
print(f"  Group 4 (80–100): {X_train_g4.shape[0]:,}")

#### Train a Random Forest Model Per Age Group

In [ ]:
print("=== Group 1: Patients aged 0–49 ===")
rf_g1 = RandomForestClassifier(max_depth=8, random_state=42)
rf_g1.fit(X_train_g1, y_train_g1)
evaluate_model(rf_g1, X_test_g1, y_test_g1)

In [ ]:
print("=== Group 2: Patients aged 50–69 ===")
rf_g2 = RandomForestClassifier(max_depth=8, random_state=42)
rf_g2.fit(X_train_g2, y_train_g2)
evaluate_model(rf_g2, X_test_g2, y_test_g2)

In [ ]:
print("=== Group 3: Patients aged 70–79 ===")
rf_g3 = RandomForestClassifier(max_depth=8, random_state=42)
rf_g3.fit(X_train_g3, y_train_g3)
evaluate_model(rf_g3, X_test_g3, y_test_g3)

In [ ]:
print("=== Group 4: Patients aged 80–100 ===")
rf_g4 = RandomForestClassifier(max_depth=8, random_state=42)
rf_g4.fit(X_train_g4, y_train_g4)
evaluate_model(rf_g4, X_test_g4, y_test_g4)

#### Age-Stratified Combined Model

The four age-specific models are assembled into a single ensemble using a routing function — effectively a `switch` that selects the appropriate sub-model for each patient based on their age group. This mirrors clinical practice, where treatment protocols often differ by patient age.

In [ ]:
def age_router(x):
    """Route each patient record to the model trained on their age group."""
    age = x['age']
    if age <= 4:
        return 0  # Group 1: 0–49
    elif age in (5, 6):
        return 1  # Group 2: 50–69
    elif age == 7:
        return 2  # Group 3: 70–79
    else:
        return 3  # Group 4: 80–100

age_ensemble = CombinedModel([rf_g1, rf_g2, rf_g3, rf_g4], age_router)

In [ ]:
pred_age      = age_ensemble.predict(X_test)
pred_age_prob = age_ensemble.predict(X_test, probabilities=True)

print("=== Age-Stratified Combined Model ===")
print(classification_report(y_test, pred_age))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, pred_age_prob), 3)}")
roc_auc(pred_age_prob, y_test)

#### Experiment 1 — Observations

The age-stratified approach produced some useful findings. Models trained on younger age groups (0–49 and 50–69) achieved meaningfully better performance than the single model trained on all patients — consistent with our initial observation that older patients exhibit more complex clinical patterns that are harder to predict. Performance declined progressively for older groups, suggesting that additional clinical features or external data may be needed to reliably predict readmission risk in elderly patients. The combined ensemble did not surpass the baseline overall, but validates the age-related performance trend noted in our earlier results section.

---

## Experiment 2 — Cluster-Based Classification

Rather than segmenting patients by a single clinical attribute like age, this experiment uses KMeans clustering to let the data itself reveal natural subgroups. The hypothesis is that patients with similar feature profiles may share readmission risk patterns that a cluster-specific model can capture better than a single global model.

In [ ]:
# Fit KMeans on training data with 3 clusters
# 3 clusters selected to balance interpretability with coverage
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_train)

# Assign cluster labels to training and test records
train_clusters = kmeans.predict(X_train)
test_clusters  = kmeans.predict(X_test)

print("Training records per cluster:")
for i in range(3):
    print(f"  Cluster {i+1}: {(train_clusters == i).sum():,}")

In [ ]:
# Partition data by cluster assignment
X_train_c1, y_train_c1 = X_train[train_clusters == 0], y_train[train_clusters == 0]
X_test_c1,  y_test_c1  = X_test[test_clusters == 0],   y_test[test_clusters == 0]

X_train_c2, y_train_c2 = X_train[train_clusters == 1], y_train[train_clusters == 1]
X_test_c2,  y_test_c2  = X_test[test_clusters == 1],   y_test[test_clusters == 1]

X_train_c3, y_train_c3 = X_train[train_clusters == 2], y_train[train_clusters == 2]
X_test_c3,  y_test_c3  = X_test[test_clusters == 2],   y_test[test_clusters == 2]

#### Train a Random Forest Model Per Cluster

In [ ]:
print("=== Cluster 1 ===")
rf_c1 = RandomForestClassifier(max_depth=8, random_state=42)
rf_c1.fit(X_train_c1, y_train_c1)
evaluate_model(rf_c1, X_test_c1, y_test_c1)

In [ ]:
print("=== Cluster 2 ===")
rf_c2 = RandomForestClassifier(max_depth=8, random_state=42)
rf_c2.fit(X_train_c2, y_train_c2)
evaluate_model(rf_c2, X_test_c2, y_test_c2)

In [ ]:
print("=== Cluster 3 ===")
rf_c3 = RandomForestClassifier(max_depth=8, random_state=42)
rf_c3.fit(X_train_c3, y_train_c3)
evaluate_model(rf_c3, X_test_c3, y_test_c3)

#### Cluster-Based Combined Model

The same ensemble structure from Experiment 1 is reused here, with the router replaced by the KMeans model — each incoming patient is assigned to the cluster whose centroid is closest, and the corresponding sub-model generates the prediction.

In [ ]:
def cluster_router(x):
    """Route each patient record to the model trained on their KMeans cluster."""
    return kmeans.predict([x])[0]

cluster_ensemble = CombinedModel([rf_c1, rf_c2, rf_c3], cluster_router)

In [ ]:
pred_cluster      = cluster_ensemble.predict(X_test)
pred_cluster_prob = cluster_ensemble.predict(X_test, probabilities=True)

print("=== Cluster-Based Combined Model ===")
print(classification_report(y_test, pred_cluster))
print(f"ROC-AUC Score: {round(roc_auc_score(y_test, pred_cluster_prob), 3)}")
roc_auc(pred_cluster_prob, y_test)

#### Experiment 2 — Observations

The cluster-based ensemble produced results very comparable to the baseline model. Performance across all three clusters was nearly uniform, which suggests the data does not separate cleanly into distinct subgroups with meaningfully different readmission patterns. This is consistent with the broader challenge highlighted throughout the project — the dataset may not contain enough discriminating clinical features to decisively separate high- and low-risk patients through feature-based subgrouping alone.

---

## Overall Summary

These two complementary experiments explored whether subgroup-specific modeling can improve readmission prediction beyond the baseline Random Forest (ROC-AUC = 0.65).

Key takeaways:

- **Age stratification** confirmed that younger patients (under 70) are easier to model than older ones. Group-specific models for those cohorts outperformed the global baseline, consistent with findings noted in related clinical ML literature.
- **Cluster-based stratification** did not improve results, suggesting the patient subgroups in this dataset are not clearly separable by their feature profiles alone.
- Both approaches highlight the same underlying limitation: accurate readmission prediction for diabetic patients — especially older ones — likely requires richer clinical variables (e.g., lab trends, discharge quality metrics, outpatient follow-up data) beyond what this dataset provides.

For Capstone 2, we plan to investigate advanced feature engineering, model fine-tuning, and potential integration of external data sources to push performance further and move toward a system that healthcare providers could realistically use to flag at-risk patients.